In [ ]:
# ============================================================================
# CELL 1: Install Packages
# ============================================================================
!pip install torch transformers==4.47.1 accelerate==0.26.1 scikit-learn numpy pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.9/270.9 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 128.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: accelerate
    Found existing installation

In [ ]:
# ============================================================================
# CELL 2: Imports
# ============================================================================
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import random
import re
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Using device: cuda
GPU: Tesla T4
GPU Memory: 14.56 GB


In [ ]:
# ============================================================================
# CELL 3: Load Data
# ============================================================================
train_path = "train_data (1).json"
test_path = "test_data_subtask_1 (1).json"

with open(train_path, 'r') as f:
    train_data = json.load(f)

with open(test_path, 'r') as f:
    test_data = json.load(f)

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nValidity distribution:\n{train_df['validity'].value_counts()}")
print(f"\nPlausibility distribution:\n{train_df['plausibility'].value_counts()}")

# Analyze 4 conditions
conditions = train_df.groupby(['plausibility', 'validity']).size()
print(f"\n4-Condition Distribution (Original):")
print(conditions)

Training samples: 960
Test samples: 191

Validity distribution:
validity
False    480
True     480
Name: count, dtype: int64

Plausibility distribution:
plausibility
False    486
True     474
Name: count, dtype: int64

4-Condition Distribution (Original):
plausibility  validity
False         False       246
              True        240
True          False       234
              True        240
dtype: int64


In [ ]:
# ============================================================================
# CELL 4: PROPER Train/Val Split BEFORE Augmentation
# ============================================================================
print("\n" + "="*80)
print("Step 1: Split ORIGINAL Data First (Prevent Data Leakage)")
print("="*80)

# Create condition column for stratification
train_df['condition'] = (
    train_df['plausibility'].astype(str) + '_' +
    train_df['validity'].astype(str)
)

# Split ORIGINAL data first
train_indices, val_indices = train_test_split(
    range(len(train_df)),
    test_size=0.15,
    random_state=42,
    stratify=train_df['condition'].values
)

train_df_original = train_df.iloc[train_indices].copy()
val_df_original = train_df.iloc[val_indices].copy()

print(f"\nOriginal data split:")
print(f"  Training: {len(train_df_original)}")
print(f"  Validation: {len(val_df_original)}")
print(f"\n CRITICAL: Augmentation will ONLY be applied to training data")
print(f"   Validation stays ORIGINAL to test true generalization")


Step 1: Split ORIGINAL Data First (Prevent Data Leakage)

Original data split:
  Training: 816
  Validation: 144

 CRITICAL: Augmentation will ONLY be applied to training data
   Validation stays ORIGINAL to test true generalization


In [ ]:
# ============================================================================
# CELL 5: Template-Based Augmentation (TRAINING ONLY)
# ============================================================================
print("\n" + "="*80)
print("Step 2: Augment ONLY Training Data")
print("="*80)

class TemplateAugmenter:
    """Generate variations that preserve logical structure"""

    def __init__(self):
        self.entity_pools = {
            'vehicles': ['car', 'bicycle', 'motorcycle', 'truck', 'bus', 'train', 'airplane', 'boat'],
            'buildings': ['house', 'building', 'tower', 'castle', 'barn', 'shed', 'cabin', 'mansion'],
            'animals': ['dog', 'cat', 'horse', 'bird', 'fish', 'rabbit', 'lion', 'tiger'],
            'people': ['person', 'human', 'individual', 'citizen', 'student', 'teacher', 'worker', 'doctor'],
            'objects': ['book', 'table', 'chair', 'tool', 'device', 'machine', 'instrument', 'gadget'],
            'food': ['fruit', 'vegetable', 'meat', 'drink', 'meal', 'snack', 'dish', 'dessert'],
            'nature': ['tree', 'plant', 'flower', 'rock', 'mountain', 'river', 'ocean', 'forest'],
        }

    def identify_entities(self, text):
        """Extract main entities"""
        entities = []
        text_lower = text.lower()

        for category, entity_list in self.entity_pools.items():
            for entity in entity_list:
                if entity in text_lower:
                    entities.append((entity, category))

        return list(set(entities))[:2]  # Max 2 entities

    def replace_entities(self, text, old_entities, new_entities):
        """Case-preserving replacement"""
        result = text

        for (old, _), (new, _) in zip(old_entities, new_entities):
            result = re.sub(re.escape(old), new, result, flags=re.IGNORECASE)
            result = re.sub(re.escape(old.capitalize()), new.capitalize(), result)

        return result

    def generate_variations(self, text, validity, plausibility, n_variations=2):
        """Generate n variations with different entities"""
        variations = []
        entities = self.identify_entities(text)

        if not entities:
            return variations

        for i in range(n_variations):
            new_entities = []

            for old_entity, old_category in entities:
                # Pick new entity from same category
                available = [e for e in self.entity_pools[old_category] if e != old_entity]
                if available:
                    new_entity = random.choice(available)
                    new_entities.append((new_entity, old_category))
                else:
                    new_entities.append((old_entity, old_category))

            if len(new_entities) == len(entities):
                new_text = self.replace_entities(text, entities, new_entities)

                if new_text != text:
                    variations.append({
                        'syllogism': new_text,
                        'validity': validity,
                        'plausibility': plausibility
                    })

        return variations

# Augment ONLY training data
augmenter = TemplateAugmenter()
augmented_train = []
variation_count = 0

for idx, row in tqdm(train_df_original.iterrows(), total=len(train_df_original), desc="Augmenting training"):
    # Add original
    augmented_train.append(row.to_dict())

    # Generate 2 variations
    variations = augmenter.generate_variations(
        row['syllogism'],
        row['validity'],
        row['plausibility'],
        n_variations=2
    )

    augmented_train.extend(variations)
    variation_count += len(variations)

train_df_augmented = pd.DataFrame(augmented_train)
train_df_augmented = train_df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✓ Training data augmentation:")
print(f"  Original training: {len(train_df_original)}")
print(f"  Generated variations: {variation_count}")
print(f"  Total training: {len(train_df_augmented)}")
print(f"  Expansion: {len(train_df_augmented)/len(train_df_original):.2f}x")

print(f"\n✓ Validation data (UNCHANGED):")
print(f"  Validation samples: {len(val_df_original)}")
print(f"  Status: NO AUGMENTATION (true generalization test)")


Step 2: Augment ONLY Training Data


Augmenting training: 100%|██████████| 816/816 [00:00<00:00, 7937.46it/s]


✓ Training data augmentation:
  Original training: 816
  Generated variations: 1224
  Total training: 2040
  Expansion: 2.50x

✓ Validation data (UNCHANGED):
  Validation samples: 144
  Status: NO AUGMENTATION (true generalization test)


In [ ]:
# ============================================================================
# CELL 6: Oversample Implausible in Training Only
# ============================================================================
print("\n" + "="*80)
print("Step 3: Oversample Implausible (Training Only)")
print("="*80)

# Separate by plausibility in augmented training data
plausible_train = train_df_augmented[train_df_augmented['plausibility'] == True]
implausible_train = train_df_augmented[train_df_augmented['plausibility'] == False]

print(f"\nBefore oversampling:")
print(f"  Plausible: {len(plausible_train)}")
print(f"  Implausible: {len(implausible_train)}")

# Oversample implausible 2x
implausible_oversampled = pd.concat([implausible_train, implausible_train], ignore_index=True)

# Combine
train_df_final = pd.concat([plausible_train, implausible_oversampled], ignore_index=True)
train_df_final = train_df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nAfter oversampling:")
print(f"  Total training: {len(train_df_final)}")

# Add dummy counterfactuals
train_df_final['counterfactual'] = train_df_final['syllogism']
val_df_original['counterfactual'] = val_df_original['syllogism']

# Final data
train_df = train_df_final
val_df = val_df_original

conditions_train = train_df.groupby(['plausibility', 'validity']).size()
conditions_val = val_df.groupby(['plausibility', 'validity']).size()

print(f"\nFinal 4-Condition Distribution:")
print(f"  TRAINING (augmented + oversampled):")
for cond, count in conditions_train.items():
    print(f"    {cond}: {count}")
print(f"  VALIDATION (original, no augmentation):")
for cond, count in conditions_val.items():
    print(f"    {cond}: {count}")

print(f"\n KEY: Validation = original data → Tests true generalization")


Step 3: Oversample Implausible (Training Only)

Before oversampling:
  Plausible: 1035
  Implausible: 1005

After oversampling:
  Total training: 3045

Final 4-Condition Distribution:
  TRAINING (augmented + oversampled):
    (False, False): 1054
    (False, True): 956
    (True, False): 529
    (True, True): 506
  VALIDATION (original, no augmentation):
    (False, False): 37
    (False, True): 36
    (True, False): 35
    (True, True): 36

 KEY: Validation = original data → Tests true generalization


In [ ]:
# ============================================================================
# CELL 7: Dataset Class
# ============================================================================
class SyllogismDataset(Dataset):
    def __init__(self, texts, counterfactuals, validity_labels, plausibility_labels, tokenizer, max_length=512):
        self.texts = texts
        self.counterfactuals = counterfactuals
        self.validity_labels = validity_labels
        self.plausibility_labels = plausibility_labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded_orig = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        encoded_cf = self.tokenizer(
            self.counterfactuals[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids_orig': encoded_orig['input_ids'].squeeze(0),
            'attention_mask_orig': encoded_orig['attention_mask'].squeeze(0),
            'input_ids_cf': encoded_cf['input_ids'].squeeze(0),
            'attention_mask_cf': encoded_cf['attention_mask'].squeeze(0),
            'validity_label': torch.tensor(self.validity_labels[idx], dtype=torch.long),
            'plausibility_label': torch.tensor(self.plausibility_labels[idx], dtype=torch.long)
        }

In [ ]:
# ============================================================================
# CELL 8: Focal Loss
# ============================================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
     inputs = inputs.float()  # ensure float32
     ce_loss = F.cross_entropy(inputs, targets, reduction='none')
     ce_loss = torch.clamp(ce_loss, max=100.0)  # prevent exp overflow
     p_t = torch.exp(-ce_loss)
     focal_loss = (1 - p_t) ** self.gamma * ce_loss

     if self.alpha is not None:
        alpha_t = self.alpha[targets]
        focal_loss = alpha_t * focal_loss

     if self.reduction == 'mean':
        return focal_loss.mean()
     elif self.reduction == 'sum':
        return focal_loss.sum()
     else:
        return focal_loss

In [ ]:
# ============================================================================
# CELL 9: Gradient Reversal Layer
# ============================================================================
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_=1.0):
        super(GradientReversalLayer, self).__init__()
        self.lambda_ = lambda_

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)

In [ ]:
# ============================================================================
# CELL 10: MLA-CI Model
# ============================================================================
class MLACIModel(nn.Module):
    def __init__(self, model_name, lambda_adv=1.0, dropout=0.2):
        super(MLACIModel, self).__init__()
        self.encoder = AutoModel.from_pretrained(
              model_name,
              output_hidden_states=True,
             low_cpu_mem_usage=False  # Force eager loading, disables accelerate's lazy materialization
             )
        self.encoder = self.encoder.float()  # Cast AFTER loading, not during

        hidden_size = self.encoder.config.hidden_size
        self.combined_size = hidden_size * 3
        self.grl = GradientReversalLayer(lambda_=lambda_adv)

        self.validity_classifier = nn.Sequential(
            nn.Linear(self.combined_size, 768),
            nn.LayerNorm(768),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

        self.plausibility_adversary = nn.Sequential(
            nn.Linear(self.combined_size, 768),
            nn.LayerNorm(768),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

    def extract_multi_layer_features(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.hidden_states

        layer_2 = hidden_states[2][:, 0, :]
        layer_6 = hidden_states[6][:, 0, :]
        layer_neg2 = hidden_states[-2][:, 0, :]

        combined = torch.cat([layer_2, layer_6, layer_neg2], dim=1)
        return combined.float()

    def forward(self, input_ids, attention_mask, return_features=False):
        features = self.extract_multi_layer_features(input_ids, attention_mask)

        validity_logits = self.validity_classifier(features)

        reversed_features = self.grl(features)
        plausibility_logits = self.plausibility_adversary(reversed_features)

        if return_features:
            return validity_logits, plausibility_logits, features
        return validity_logits, plausibility_logits

In [ ]:
# ============================================================================
# CELL 11: Data Preparation (Using Pre-Split Data)
# ============================================================================
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Use the already-split data (NO SECOND SPLIT!)
# train_df = augmented + oversampled training data
# val_df = original validation data (no augmentation)

print(f"\n Using Pre-Split Data:")
print(f"  Training: {len(train_df)} samples (augmented + oversampled)")
print(f"  Validation: {len(val_df)} samples (original, no augmentation)")

# Prepare training data
train_texts = train_df['syllogism'].tolist()
train_cf = train_df['counterfactual'].tolist()
train_validity = train_df['validity'].astype(int).values
train_plausibility = train_df['plausibility'].astype(int).values

# Prepare validation data
val_texts = val_df['syllogism'].tolist()
val_cf = val_df['counterfactual'].tolist()
val_validity = val_df['validity'].astype(int).values
val_plausibility = val_df['plausibility'].astype(int).values

# Create datasets
train_dataset = SyllogismDataset(train_texts, train_cf, train_validity, train_plausibility, tokenizer)
val_dataset = SyllogismDataset(val_texts, val_cf, val_validity, val_plausibility, tokenizer)

# Create dataloaders
BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Batch size: {BATCH_SIZE}")

# Initialize model
model = MLACIModel(MODEL_NAME, lambda_adv=1.0, dropout=0.2)
model = model.to(device)

print(f"  Model parameters: {sum(p.numel() for p in model.parameters()):,}")

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]


 Using Pre-Split Data:
  Training: 3045 samples (augmented + oversampled)
  Validation: 144 samples (original, no augmentation)

DataLoaders created:
  Train batches: 762
  Val batches: 36
  Batch size: 4


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

  Model parameters: 187,769,860


In [ ]:
# ============================================================================
# MULTI-SEED EXPERIMENTS
# Experiment B: −Adversarial (seeds 43, 44) — we already have seed 42
# Experiment C: Full system (seeds 43, 44) — we already have seed 42
# ============================================================================
import json
import gc
import numpy as np
from sklearn.metrics import accuracy_score

# Verify data is ready from cells 1-11
print("Data verification:")
print(f"  full_train (aug+oversample): {len(train_df)} samples")
print(f"  val_df_original: {len(val_df_original)} samples")
print(f"\nWill run 4 experiments (~2 hours total):")
print(f"  1. −Adversarial, seed 43")
print(f"  2. −Adversarial, seed 44")
print(f"  3. Full MLA-CI, seed 43")
print(f"  4. Full MLA-CI, seed 44")

Data verification:
  full_train (aug+oversample): 3045 samples
  val_df_original: 144 samples

Will run 4 experiments (~2 hours total):
  1. −Adversarial, seed 43
  2. −Adversarial, seed 44
  3. Full MLA-CI, seed 43
  4. Full MLA-CI, seed 44


In [ ]:
# ============================================================================
# Reusable training + evaluation function
# ============================================================================
def run_seeded_experiment(
    experiment_name,
    seed,
    train_df_exp,
    val_df_exp,
    model_class,
    lambda_adv,
    use_focal,
    focal_gamma=2.0,
    epochs=4,
    lr=2e-5,
    batch_size=4,
    accumulation_steps=4,
):
    """Run one full experiment with a specific seed."""

    print(f"\n{'#'*80}")
    print(f"# {experiment_name} | Seed={seed}")
    print(f"{'#'*80}")

    # Set seed AFTER data prep — only affects model init + training randomness
    set_seed(seed)

    # Ensure counterfactual column
    tr_df = train_df_exp.copy()
    vl_df = val_df_exp.copy()
    if 'counterfactual' not in tr_df.columns:
        tr_df['counterfactual'] = tr_df['syllogism']
    if 'counterfactual' not in vl_df.columns:
        vl_df['counterfactual'] = vl_df['syllogism']

    # Prepare data
    tr_texts = tr_df['syllogism'].tolist()
    tr_cf = tr_df['counterfactual'].tolist()
    tr_validity = tr_df['validity'].astype(int).values
    tr_plausibility = tr_df['plausibility'].astype(int).values

    v_texts = vl_df['syllogism'].tolist()
    v_cf = vl_df['counterfactual'].tolist()
    v_validity = vl_df['validity'].astype(int).values
    v_plausibility = vl_df['plausibility'].astype(int).values

    # Datasets & loaders
    tr_dataset = SyllogismDataset(tr_texts, tr_cf, tr_validity, tr_plausibility, tokenizer)
    v_dataset = SyllogismDataset(v_texts, v_cf, v_validity, v_plausibility, tokenizer)
    tr_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    v_loader = DataLoader(v_dataset, batch_size=batch_size, shuffle=False)

    print(f"  Train: {len(tr_df)} | Val: {len(vl_df)} | λ_adv={lambda_adv} | Focal={use_focal}")

    # Model
    exp_model = model_class(MODEL_NAME, lambda_adv=lambda_adv, dropout=0.2)
    exp_model = exp_model.to(device)

    # Loss
    if use_focal:
        val_criterion = FocalLoss(gamma=focal_gamma, alpha=None)
    else:
        val_criterion = nn.CrossEntropyLoss()
    adv_criterion = nn.CrossEntropyLoss()

    # Optimizer & scheduler
    exp_optimizer = torch.optim.AdamW(exp_model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = (len(tr_loader) // accumulation_steps) * epochs
    exp_scheduler = get_linear_schedule_with_warmup(
        exp_optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    best_score = 0
    best_results = {}
    epoch_results = []

    for epoch in range(epochs):
        # --- TRAIN ---
        exp_model.train()
        correct = 0
        total = 0
        exp_optimizer.zero_grad()

        for batch_idx, batch in enumerate(tqdm(tr_loader, desc=f"[Seed {seed}] Epoch {epoch+1}")):
            input_ids = batch['input_ids_orig'].to(device)
            attention_mask = batch['attention_mask_orig'].to(device)
            val_labels = batch['validity_label'].to(device)
            plaus_labels = batch['plausibility_label'].to(device)

            validity_logits, plausibility_logits = exp_model(input_ids, attention_mask)

            loss_val = val_criterion(validity_logits, val_labels)
            loss_plaus = adv_criterion(plausibility_logits, plaus_labels)
            loss = (loss_val + lambda_adv * loss_plaus) / accumulation_steps

            loss.backward()

            if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(tr_loader):
                torch.nn.utils.clip_grad_norm_(exp_model.parameters(), 1.0)
                exp_optimizer.step()
                exp_scheduler.step()
                exp_optimizer.zero_grad()

            preds = torch.argmax(validity_logits, dim=1)
            correct += (preds == val_labels).sum().item()
            total += val_labels.size(0)

        train_acc = correct / total

        # --- EVALUATE ---
        exp_model.eval()
        all_preds = []
        all_val = []
        all_plaus = []
        idx = 0

        with torch.no_grad():
            for batch in v_loader:
                input_ids = batch['input_ids_orig'].to(device)
                attention_mask = batch['attention_mask_orig'].to(device)
                val_labels = batch['validity_label'].to(device)
                bs = val_labels.size(0)

                validity_logits, _ = exp_model(input_ids, attention_mask)
                preds = torch.argmax(validity_logits, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_val.extend(val_labels.cpu().numpy())
                for i in range(bs):
                    all_plaus.append(v_plausibility[idx])
                    idx += 1

        all_preds = np.array(all_preds)
        all_val = np.array(all_val)
        all_plaus = np.array(all_plaus)

        val_acc = accuracy_score(all_val, all_preds)

        # Per-condition
        conditions = {}
        for plaus in [0, 1]:
            for valid in [0, 1]:
                mask = (all_plaus == plaus) & (all_val == valid)
                if mask.sum() > 0:
                    acc = accuracy_score(all_val[mask], all_preds[mask])
                    plaus_name = "Plausible" if plaus == 1 else "Implausible"
                    valid_name = "Valid" if valid == 1 else "Invalid"
                    conditions[f"{plaus_name}_{valid_name}"] = acc

        # Content effect
        intra_plaus = abs(conditions.get('Plausible_Valid', 0) - conditions.get('Plausible_Invalid', 0))
        intra_implaus = abs(conditions.get('Implausible_Valid', 0) - conditions.get('Implausible_Invalid', 0))
        intra_effect = (intra_plaus + intra_implaus) / 2
        cross_valid = abs(conditions.get('Plausible_Valid', 0) - conditions.get('Implausible_Valid', 0))
        cross_invalid = abs(conditions.get('Plausible_Invalid', 0) - conditions.get('Implausible_Invalid', 0))
        cross_effect = (cross_valid + cross_invalid) / 2
        val_ce = (intra_effect + cross_effect) / 2

        combined = val_acc * 100 / (1 + np.log(1 + val_ce * 100))

        print(f"  Epoch {epoch+1}: Acc={val_acc*100:.2f}% | CE={val_ce*100:.2f}% | Score={combined:.2f}")

        epoch_results.append({
            'epoch': epoch + 1,
            'train_acc': round(train_acc * 100, 2),
            'val_acc': round(val_acc * 100, 2),
            'val_ce': round(val_ce * 100, 2),
            'combined_score': round(combined, 2),
            'conditions': {k: round(v * 100, 2) for k, v in conditions.items()}
        })

        if combined > best_score:
            best_score = combined
            best_results = {
                'experiment': experiment_name,
                'seed': seed,
                'best_epoch': epoch + 1,
                'val_acc': round(val_acc * 100, 2),
                'val_ce': round(val_ce * 100, 2),
                'combined_score': round(combined, 2),
                'conditions': {k: round(v * 100, 2) for k, v in conditions.items()},
            }

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    best_results['all_epochs'] = epoch_results

    # Cleanup
    del exp_model, exp_optimizer, exp_scheduler, tr_loader, v_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"\n  ★ BEST: Epoch {best_results['best_epoch']} | Acc={best_results['val_acc']}% | CE={best_results['val_ce']}% | Score={best_results['combined_score']}")
    return best_results

print("✓ Training function defined")

✓ Training function defined


In [ ]:
# ============================================================================
# RUN ALL 4 EXPERIMENTS
# ============================================================================

# Use full pipeline data (aug + oversample) for both configs
full_train_data = train_df.copy()  # 3045 samples
val_data = val_df_original.copy()  # 144 samples

multi_seed_results = {}

# ── Experiment B1: −Adversarial, seed 43 ──
multi_seed_results['no_adv_seed43'] = run_seeded_experiment(
    experiment_name="− Adversarial (λ=0)",
    seed=43,
    train_df_exp=full_train_data,
    val_df_exp=val_data,
    model_class=MLACIModel,
    lambda_adv=0.0,
    use_focal=True,
    epochs=4,
)

# ── Experiment B2: −Adversarial, seed 44 ──
multi_seed_results['no_adv_seed44'] = run_seeded_experiment(
    experiment_name="− Adversarial (λ=0)",
    seed=44,
    train_df_exp=full_train_data,
    val_df_exp=val_data,
    model_class=MLACIModel,
    lambda_adv=0.0,
    use_focal=True,
    epochs=4,
)

# ── Experiment C1: Full system, seed 43 ──
multi_seed_results['full_seed43'] = run_seeded_experiment(
    experiment_name="Full MLA-CI",
    seed=43,
    train_df_exp=full_train_data,
    val_df_exp=val_data,
    model_class=MLACIModel,
    lambda_adv=1.0,
    use_focal=True,
    epochs=4,
)

# ── Experiment C2: Full system, seed 44 ──
multi_seed_results['full_seed44'] = run_seeded_experiment(
    experiment_name="Full MLA-CI",
    seed=44,
    train_df_exp=full_train_data,
    val_df_exp=val_data,
    model_class=MLACIModel,
    lambda_adv=1.0,
    use_focal=True,
    epochs=4,
)

print("\n\n All 4 experiments complete!")


################################################################################
# − Adversarial (λ=0) | Seed=43
################################################################################
  Train: 3045 | Val: 144 | λ_adv=0.0 | Focal=True


[Seed 43] Epoch 1: 100%|██████████| 762/762 [06:32<00:00,  1.94it/s]


  Epoch 1: Acc=84.03% | CE=5.71% | Score=28.95


[Seed 43] Epoch 2: 100%|██████████| 762/762 [06:42<00:00,  1.90it/s]


  Epoch 2: Acc=90.28% | CE=4.37% | Score=33.69


...

  Epoch 2: Acc=72.22% | CE=27.14% | Score=16.65


[Seed 44] Epoch 3: 100%|██████████| 762/762 [06:42<00:00,  1.89it/s]


  Epoch 3: Acc=84.03% | CE=9.18% | Score=25.30


[Seed 44] Epoch 4: 100%|██████████| 762/762 [06:42<00:00,  1.89it/s]


  Epoch 4: Acc=85.42% | CE=9.26% | Score=25.66

  ★ BEST: Epoch 4 | Acc=85.42% | CE=9.26% | Score=25.66


✅ All 4 experiments complete!


In [ ]:
# ============================================================================
# COMPUTE MEAN ± STD AND SAVE
# ============================================================================

# Seed 42 results (from previous experiments)
seed42_no_adv = {'val_acc': 93.06, 'val_ce': 4.20, 'combined_score': 35.12}
seed42_full   = {'val_acc': 83.33, 'val_ce': 8.33, 'combined_score': 25.77}

# Collect −Adversarial across 3 seeds
no_adv_accs = [
    seed42_no_adv['val_acc'],
    multi_seed_results['no_adv_seed43']['val_acc'],
    multi_seed_results['no_adv_seed44']['val_acc']
]
no_adv_ces = [
    seed42_no_adv['val_ce'],
    multi_seed_results['no_adv_seed43']['val_ce'],
    multi_seed_results['no_adv_seed44']['val_ce']
]
no_adv_scores = [
    seed42_no_adv['combined_score'],
    multi_seed_results['no_adv_seed43']['combined_score'],
    multi_seed_results['no_adv_seed44']['combined_score']
]

# Collect Full system across 3 seeds
full_accs = [
    seed42_full['val_acc'],
    multi_seed_results['full_seed43']['val_acc'],
    multi_seed_results['full_seed44']['val_acc']
]
full_ces = [
    seed42_full['val_ce'],
    multi_seed_results['full_seed43']['val_ce'],
    multi_seed_results['full_seed44']['val_ce']
]
full_scores = [
    seed42_full['combined_score'],
    multi_seed_results['full_seed43']['combined_score'],
    multi_seed_results['full_seed44']['combined_score']
]

# Print detailed results
print("=" * 80)
print("MULTI-SEED RESULTS")
print("=" * 80)

print(f"\n{'−Adversarial (λ=0) across 3 seeds:'}")
print(f"  {'Seed':<8} {'Acc':<10} {'CE':<10} {'Score':<10}")
print(f"  {'-'*38}")
for i, seed in enumerate([42, 43, 44]):
    print(f"  {seed:<8} {no_adv_accs[i]:<10} {no_adv_ces[i]:<10} {no_adv_scores[i]:<10}")
print(f"  {'-'*38}")
print(f"  {'Mean':<8} {np.mean(no_adv_accs):<10.2f} {np.mean(no_adv_ces):<10.2f} {np.mean(no_adv_scores):<10.2f}")
print(f"  {'Std':<8} {np.std(no_adv_accs):<10.2f} {np.std(no_adv_ces):<10.2f} {np.std(no_adv_scores):<10.2f}")

print(f"\n{'Full MLA-CI across 3 seeds:'}")
print(f"  {'Seed':<8} {'Acc':<10} {'CE':<10} {'Score':<10}")
print(f"  {'-'*38}")
for i, seed in enumerate([42, 43, 44]):
    print(f"  {seed:<8} {full_accs[i]:<10} {full_ces[i]:<10} {full_scores[i]:<10}")
print(f"  {'-'*38}")
print(f"  {'Mean':<8} {np.mean(full_accs):<10.2f} {np.mean(full_ces):<10.2f} {np.mean(full_scores):<10.2f}")
print(f"  {'Std':<8} {np.std(full_accs):<10.2f} {np.std(full_ces):<10.2f} {np.std(full_scores):<10.2f}")

# Key comparison
print(f"\n{'=' * 80}")
print("KEY COMPARISON (mean ± std)")
print(f"{'=' * 80}")
print(f"  −Adversarial: Score = {np.mean(no_adv_scores):.2f} ± {np.std(no_adv_scores):.2f}")
print(f"  Full MLA-CI:  Score = {np.mean(full_scores):.2f} ± {np.std(full_scores):.2f}")
gap = np.mean(no_adv_scores) - np.mean(full_scores)
print(f"  Gap: +{gap:.2f} points (removing adversarial helps)")

# Check if the confidence intervals overlap
no_adv_low = np.mean(no_adv_scores) - np.std(no_adv_scores)
full_high = np.mean(full_scores) + np.std(full_scores)
if no_adv_low > full_high:
    print(f"   Non-overlapping ranges — finding is robust!")
else:
    print(f"   Ranges overlap slightly — but directional finding likely holds")

# Save everything
save_data = {
    'no_adv_seeds': {
        'seed_42': seed42_no_adv,
        'seed_43': {
            'val_acc': multi_seed_results['no_adv_seed43']['val_acc'],
            'val_ce': multi_seed_results['no_adv_seed43']['val_ce'],
            'combined_score': multi_seed_results['no_adv_seed43']['combined_score'],
            'conditions': multi_seed_results['no_adv_seed43']['conditions'],
        },
        'seed_44': {
            'val_acc': multi_seed_results['no_adv_seed44']['val_acc'],
            'val_ce': multi_seed_results['no_adv_seed44']['val_ce'],
            'combined_score': multi_seed_results['no_adv_seed44']['combined_score'],
            'conditions': multi_seed_results['no_adv_seed44']['conditions'],
        },
        'mean_acc': round(np.mean(no_adv_accs), 2),
        'std_acc': round(np.std(no_adv_accs), 2),
        'mean_ce': round(np.mean(no_adv_ces), 2),
        'std_ce': round(np.std(no_adv_ces), 2),
        'mean_score': round(np.mean(no_adv_scores), 2),
        'std_score': round(np.std(no_adv_scores), 2),
    },
    'full_seeds': {
        'seed_42': seed42_full,
        'seed_43': {
            'val_acc': multi_seed_results['full_seed43']['val_acc'],
            'val_ce': multi_seed_results['full_seed43']['val_ce'],
            'combined_score': multi_seed_results['full_seed43']['combined_score'],
            'conditions': multi_seed_results['full_seed43']['conditions'],
        },
        'seed_44': {
            'val_acc': multi_seed_results['full_seed44']['val_acc'],
            'val_ce': multi_seed_results['full_seed44']['val_ce'],
            'combined_score': multi_seed_results['full_seed44']['combined_score'],
            'conditions': multi_seed_results['full_seed44']['conditions'],
        },
        'mean_acc': round(np.mean(full_accs), 2),
        'std_acc': round(np.std(full_accs), 2),
        'mean_ce': round(np.mean(full_ces), 2),
        'std_ce': round(np.std(full_ces), 2),
        'mean_score': round(np.mean(full_scores), 2),
        'std_score': round(np.std(full_scores), 2),
    },
    'raw_results': {k: {
        'val_acc': v['val_acc'],
        'val_ce': v['val_ce'],
        'combined_score': v['combined_score'],
        'best_epoch': v['best_epoch'],
        'conditions': v['conditions'],
    } for k, v in multi_seed_results.items()}
}

with open('multi_seed_results.json', 'w') as f:
    json.dump(save_data, f, indent=2)

print(f"\n Saved to multi_seed_results.json")
print(f"\nFor the paper, report:")
print(f"  −Adversarial: {np.mean(no_adv_accs):.1f}±{np.std(no_adv_accs):.1f} acc, {np.mean(no_adv_ces):.1f}±{np.std(no_adv_ces):.1f} CE, {np.mean(no_adv_scores):.2f}±{np.std(no_adv_scores):.2f} score")
print(f"  Full MLA-CI:  {np.mean(full_accs):.1f}±{np.std(full_accs):.1f} acc, {np.mean(full_ces):.1f}±{np.std(full_ces):.1f} CE, {np.mean(full_scores):.2f}±{np.std(full_scores):.2f} score")

MULTI-SEED RESULTS

−Adversarial (λ=0) across 3 seeds:
  Seed     Acc        CE         Score     
  --------------------------------------
  42       93.06      4.2        35.12     
  43       90.28      4.37       33.69     
  44       90.28      1.66       45.63     
  --------------------------------------
  Mean     91.21      3.41       38.15     
  Std      1.31       1.24       5.32      

Full MLA-CI across 3 seeds:
  Seed     Acc        CE         Score     
  --------------------------------------
  42       83.33      8.33       25.77     
  43       78.47      5.18       27.81     
  44       85.42      9.26       25.66     
  --------------------------------------
  Mean     82.41      7.59       26.41     
  Std      2.91       1.75       0.99      

KEY COMPARISON (mean ± std)
  −Adversarial: Score = 38.15 ± 5.32
  Full MLA-CI:  Score = 26.41 ± 0.99
  Gap: +11.73 points (removing adversarial helps)
  ✓ Non-overlapping ranges — finding is robust!

✅ Saved to multi_seed_